
# Interactive ML Decision Lab
## From Data → Scikit-Learn Pipeline → GridSearchCV → Matplotlib → LLM Decision Support

This notebook demonstrates a complete, production-shaped ML workflow using **scikit-learn** and **Matplotlib**, then shows how the winning ML model can become a structured decision tool for an LLM.

### Flow

```text
Dataset
  ↓
Explore + visualize
  ↓
Preprocess
  ↓
Train/test split
  ↓
Choose candidate models
  ↓
GridSearchCV + cross-validation
  ↓
Leaderboard
  ↓
Champion model
  ↓
Diagnostics + visualizations
  ↓
Structured ML evidence
  ↓
LLM decision/orchestration layer
```

The notebook intentionally separates:
- **ML prediction**: statistical / specialized quantitative judgment
- **LLM reasoning**: synthesis, explanation, planning, and tool orchestration


In [ ]:

# If needed, uncomment:
# %pip install scikit-learn pandas numpy matplotlib ipywidgets

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_breast_cancer, load_diabetes, load_iris, load_wine
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    RandomForestRegressor, GradientBoostingRegressor,
    IsolationForest
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, mean_squared_error, r2_score
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC, SVR

print("Environment ready.")



## 1. Pick a sample problem

You can later replace this section with your own CSV loader.

- `breast_cancer`: binary classification
- `iris`: multiclass classification
- `wine`: multiclass classification
- `diabetes`: regression


In [ ]:

DATASETS = {
    "breast_cancer": ("classification", load_breast_cancer),
    "iris": ("classification", load_iris),
    "wine": ("classification", load_wine),
    "diabetes": ("regression", load_diabetes),
}

dataset_name = "breast_cancer"   # change me

problem_type, loader = DATASETS[dataset_name]
bundle = loader(as_frame=True)

df = bundle.frame.copy()
target_col = bundle.target.name if hasattr(bundle.target, "name") and bundle.target.name else "target"

if target_col not in df.columns:
    df[target_col] = bundle.target

print("Dataset:", dataset_name)
print("Problem:", problem_type)
print("Shape:", df.shape)
display(df.head())


## 2. Inspect the data

In [ ]:

print("Target column:", target_col)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing").sort_values("missing", ascending=False).head(15))

print("\nSummary:")
display(df.describe(include="all").T.head(20))


## 3. Matplotlib: visualize the ML problem

In [ ]:

# Target distribution
plt.figure(figsize=(7, 4))

if problem_type == "classification":
    df[target_col].value_counts().sort_index().plot(kind="bar")
    plt.ylabel("Count")
    plt.title("Target Class Distribution")
else:
    plt.hist(df[target_col], bins=25)
    plt.ylabel("Frequency")
    plt.title("Target Distribution")

plt.xlabel(target_col)
plt.show()


In [ ]:

# Correlation view for numeric columns
numeric_df = df.select_dtypes(include=np.number)

corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=7)
plt.yticks(range(len(corr.columns)), corr.columns, fontsize=7)
plt.title("Numeric Feature Correlation Matrix")
plt.tight_layout()
plt.show()



## 4. Build the preprocessing pipeline

This section automatically identifies numeric and categorical features.

The same preprocessing is used during both:
- training
- future prediction

That prevents training/serving preprocessing drift.


In [ ]:

X = df.drop(columns=[target_col])
y = df[target_col]

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

print("Numeric:", numeric_features[:10], "..." if len(numeric_features) > 10 else "")
print("Categorical:", categorical_features)


## 5. Train/test split

In [ ]:

stratify = y if problem_type == "classification" else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=stratify
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)



## 6. Choose candidate models

Edit `selected_models`.

This is the important idea:

> GridSearchCV does not invent the candidate models.  
> **You define the legitimate model families, and it runs the tournament.**


In [ ]:

if problem_type == "classification":
    MODEL_SPACE = {
        "logistic_regression": {
            "model": LogisticRegression(max_iter=3000),
            "params": {
                "model__C": [0.1, 1.0, 10.0]
            }
        },
        "random_forest": {
            "model": RandomForestClassifier(random_state=42),
            "params": {
                "model__n_estimators": [100, 300],
                "model__max_depth": [None, 5, 10]
            }
        },
        "gradient_boosting": {
            "model": GradientBoostingClassifier(random_state=42),
            "params": {
                "model__n_estimators": [50, 100, 200],
                "model__learning_rate": [0.03, 0.1]
            }
        },
        "svm": {
            "model": SVC(probability=True),
            "params": {
                "model__C": [0.1, 1.0, 10.0],
                "model__kernel": ["linear", "rbf"]
            }
        },
        "knn": {
            "model": KNeighborsClassifier(),
            "params": {
                "model__n_neighbors": [3, 5, 9, 15]
            }
        },
    }
    scoring_metric = "f1_weighted"
    selected_models = ["logistic_regression", "random_forest", "gradient_boosting", "svm"]

else:
    MODEL_SPACE = {
        "linear_regression": {
            "model": LinearRegression(),
            "params": {}
        },
        "ridge": {
            "model": Ridge(),
            "params": {
                "model__alpha": [0.01, 0.1, 1.0, 10.0]
            }
        },
        "random_forest": {
            "model": RandomForestRegressor(random_state=42),
            "params": {
                "model__n_estimators": [100, 300],
                "model__max_depth": [None, 5, 10]
            }
        },
        "gradient_boosting": {
            "model": GradientBoostingRegressor(random_state=42),
            "params": {
                "model__n_estimators": [50, 100, 200],
                "model__learning_rate": [0.03, 0.1]
            }
        },
        "knn": {
            "model": KNeighborsRegressor(),
            "params": {
                "model__n_neighbors": [3, 5, 9, 15]
            }
        },
        "svr": {
            "model": SVR(),
            "params": {
                "model__C": [0.1, 1.0, 10.0],
                "model__kernel": ["linear", "rbf"]
            }
        },
    }
    scoring_metric = "neg_root_mean_squared_error"
    selected_models = ["ridge", "random_forest", "gradient_boosting", "svr"]

print("Selected:", selected_models)
print("Scoring:", scoring_metric)



## 7. Run the GridSearchCV model tournament

Each candidate receives the same preprocessing pipeline.

This means GridSearchCV evaluates not just model parameters, but the **whole training workflow** consistently.


In [ ]:

results = []
search_objects = {}

for name in selected_models:
    spec = MODEL_SPACE[name]

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", spec["model"]),
    ])

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=spec["params"],
        scoring=scoring_metric,
        cv=5,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
    )

    print(f"Training {name}...")
    search.fit(X_train, y_train)

    search_objects[name] = search

    results.append({
        "model": name,
        "best_cv_score": search.best_score_,
        "best_params": search.best_params_,
        "best_estimator": search.best_estimator_,
    })

leaderboard = pd.DataFrame(results)

if problem_type == "classification":
    leaderboard = leaderboard.sort_values("best_cv_score", ascending=False)
else:
    # GridSearchCV returns negative RMSE because larger-is-better convention
    leaderboard["cv_rmse"] = -leaderboard["best_cv_score"]
    leaderboard = leaderboard.sort_values("cv_rmse", ascending=True)

display(leaderboard.drop(columns=["best_estimator"]))


## 8. Visualize the model leaderboard

In [ ]:

plot_df = leaderboard.copy()

plt.figure(figsize=(9, 5))

if problem_type == "classification":
    plt.bar(plot_df["model"], plot_df["best_cv_score"])
    plt.ylabel(scoring_metric)
    plt.title("Cross-Validated Model Tournament")
else:
    plt.bar(plot_df["model"], plot_df["cv_rmse"])
    plt.ylabel("CV RMSE (lower is better)")
    plt.title("Cross-Validated Model Tournament")

plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## 9. Select the champion

In [ ]:

champion_row = leaderboard.iloc[0]
champion_name = champion_row["model"]
champion = champion_row["best_estimator"]

print("CHAMPION:", champion_name)
print("Best parameters:", champion_row["best_params"])


## 10. Holdout evaluation

In [ ]:

pred = champion.predict(X_test)

if problem_type == "classification":
    accuracy = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average="weighted")

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Weighted F1: {f1:.4f}")
    print("\nClassification report:\n")
    print(classification_report(y_test, pred))

    ConfusionMatrixDisplay.from_predictions(y_test, pred)
    plt.title(f"Confusion Matrix — {champion_name}")
    plt.show()

else:
    rmse = mean_squared_error(y_test, pred) ** 0.5
    r2 = r2_score(y_test, pred)

    print(f"RMSE: {rmse:.4f}")
    print(f"R²:   {r2:.4f}")

    plt.figure(figsize=(7, 5))
    plt.scatter(y_test, pred, alpha=0.7)
    low = min(y_test.min(), pred.min())
    high = max(y_test.max(), pred.max())
    plt.plot([low, high], [low, high])
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"Actual vs Predicted — {champion_name}")
    plt.show()

    residuals = y_test - pred
    plt.figure(figsize=(7, 5))
    plt.scatter(pred, residuals, alpha=0.7)
    plt.axhline(0)
    plt.xlabel("Predicted")
    plt.ylabel("Residual")
    plt.title("Residual Plot")
    plt.show()



## 11. Optional binary-classification ROC curve

This cell runs only when:
- the problem is classification,
- the target is binary,
- and the model exposes probabilities.


In [ ]:

if problem_type == "classification" and len(np.unique(y_test)) == 2 and hasattr(champion, "predict_proba"):
    prob = champion.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {champion_name}")
    plt.legend()
    plt.show()
else:
    print("ROC curve not applicable to this model/problem.")



# Part II — Turn the ML model into an AI decision tool

The next section demonstrates the architectural jump:

```text
specialized ML model
      ↓
structured prediction
      ↓
LLM context/tool result
      ↓
reasoning + explanation + next action
```

The LLM is **not secretly running scikit-learn internally**.

Instead, the production application explicitly exposes the ML model as a tool.


In [ ]:

# Pick one real holdout example
sample_index = X_test.index[0]
sample = X_test.loc[[sample_index]]
actual = y_test.loc[sample_index]

prediction = champion.predict(sample)[0]

ml_evidence = {
    "model_name": champion_name,
    "actual_value_for_demo": float(actual) if np.issubdtype(type(actual), np.number) else str(actual),
    "prediction": float(prediction) if np.issubdtype(type(prediction), np.number) else str(prediction),
}

if problem_type == "classification" and hasattr(champion, "predict_proba"):
    probs = champion.predict_proba(sample)[0]
    ml_evidence["class_probabilities"] = {
        str(cls): float(p)
        for cls, p in zip(champion.classes_, probs)
    }

ml_evidence



## 12. Build an LLM-ready evidence package

A good production pattern is to give the LLM **structured evidence**, not a vague prose summary.


In [ ]:

decision_context = {
    "task": f"{problem_type} decision support",
    "dataset": dataset_name,
    "champion_model": champion_name,
    "ml_evidence": ml_evidence,
    "instruction_to_llm": (
        "Use the ML prediction as quantitative evidence. "
        "Do not treat it as absolute truth. Explain the recommendation, "
        "state uncertainty, and identify any additional evidence needed."
    )
}

print(json.dumps(decision_context, indent=2))



## 13. Example LLM orchestration pattern

The code below is intentionally provider-neutral pseudocode.

In a real system, this could call:
- OpenAI
- Anthropic
- Gemini
- a local model served with vLLM

The important part is the **contract** between the ML system and the LLM.


In [ ]:

def build_llm_prompt(context: dict) -> str:
    return f"""
You are an AI decision-support system.

The following prediction came from a validated machine-learning model.

ML EVIDENCE
------------
{json.dumps(context["ml_evidence"], indent=2)}

TASK
----
{context["instruction_to_llm"]}

Return:
1. Interpretation
2. Recommended action
3. Confidence / uncertainty
4. Additional evidence that would improve the decision
"""

prompt = build_llm_prompt(decision_context)
print(prompt)

# Example future integration:
#
# response = llm_client.responses.create(
#     model="your-model",
#     input=prompt
# )
#
# print(response.output_text)



# Part III — Production architecture

The notebook's ML object can become a service endpoint:

```text
User / Agent
    ↓
LLM
    ↓ tool call
predict_risk(...)
    ↓
FastAPI
    ↓
Champion ML model
    ↓
prediction + probability
    ↓
LLM
    ↓
decision / explanation / action
```

A larger production version could add:

```text
Data ingestion
    ↓
Feature pipeline
    ↓
scikit-learn / PyTorch training
    ↓
GridSearchCV / tuning
    ↓
MLflow Registry
    ↓
Champion / Challenger
    ↓
FastAPI
    ↓
Docker
    ↓
Kubernetes
    ↓
LLM / Agent tool calls
    ↓
Telemetry + drift monitoring
    ↓
retraining / promotion
```

This is the bridge between **classical ML engineering** and **agentic AI systems**.



## 14. Experiments to try next

1. Switch datasets.
2. Add/remove models from `selected_models`.
3. Change the scoring metric.
4. Increase parameter grids.
5. Load your own CSV.
6. Add feature engineering.
7. Compare GridSearchCV with RandomizedSearchCV.
8. Track experiments in MLflow.
9. Serve the champion through FastAPI.
10. Give the endpoint to an LLM as a callable tool.
